In [1]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\aviru\.vscode\RAG\RAG-With-langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\aviru\AppData\Local\Temp\ipykernel_21048\983744508.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


### PDF loader

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the pdf files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ##Pattern to match files
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False
)
pdf_documents = dir_loader.load()

print(f"Loaded {len(pdf_documents)} pages")
print(pdf_documents[0].page_content)
print(pdf_documents[0].metadata)

### Split documents into chunks

In [3]:
def split_documents(documents, chunk_size=500, chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(
        f"Split {len(documents)} documents into {len(split_docs)} chunks"
    )

    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [ ]:
chunks = split_documents(
    pdf_documents,
    chunk_size=500,
    chunk_overlap=100
)


In [5]:
chunks

[Document(metadata={'producer': 'Skia/PDF m150', 'creator': 'FlowCV - https://flowcv.com', 'creationdate': '2026-07-29T14:44:08+00:00', 'source': '..\\data\\pdf\\Avirup_kar_CV.pdf', 'file_path': '..\\data\\pdf\\Avirup_kar_CV.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Avirup kar Resume', 'author': '', 'subject': '', 'keywords': 'FlowCV – Online Resume Builder – https://flowcv.com', 'moddate': '2026-07-29T14:44:08+00:00', 'trapped': '', 'modDate': "D:20260729144408+00'00'", 'creationDate': "D:20260729144408+00'00'", 'page': 0}, page_content='Avirup kar\navirupk57@gmail.com\n \nGitHub\n \nLinkedIn\n \nX.com\n \nPortfolio\n \nPROFILE\n-Full-stack developer building modern web applications with React and Next.js. I focus on clean UI, scalable \nbackend logic, and writing maintainable, performance-oriented code.\nI enjoy designing complete systems from frontend to backend while continuously improving my skills through \nreal-world projects.\nSKILLS\nFrontend & Backend Development

### embedding And vectorStoreDB

### Manage Embedding

In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
           model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer model"""  
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
            Generate embeddings for a list of texts
            Args:
            texts: List of text strings to embed
            Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
           raise ValueError("Model not loaded")
       
        print (f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print (f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings        
    
    
## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1460.98it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [7]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """             
            Initialize the vector store
            Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise    
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
            Add documents and their embeddings to the vector store
            Args:
                documents: List of LangChain documents
                embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
           raise ValueError("Number of documents must match number of embeddings")
       
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
        
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
            
            # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print (f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        

vectorStore = VectorStore()
vectorStore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 18


In [19]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

### Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorStore.add_documents(chunks, embeddings)


Generating embeddings for 6 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.12it/s]

Generated embeddings with shape: (6, 384)
Adding 6 documents to vector store...
Successfully added 6 documents to vector store
Total documents in collection: 18


### Retriever Pipeline From VectorStore


In [26]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
         Initialize the retriever
         Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
            Retrieve relevant documents for a query
            Args:
                query: The search query
                    top_k: Number of top results to return
                    score_threshold: Minimum similarity score threshold
                Returns:
                    List of dictionaries containing retrieved documents and metadata
        """
        print (f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]   
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = [] 
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'] [0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0] 
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    retrieved_docs.append({
                      "id": doc_id,
                       "content": document,
                      "metadata": metadata,
                      "distance": distance,
                      "similarity_score": similarity_score,
                      "rank": i + 1,
                    })     
                    # retrieved_docs.append({
                    #     "id": doc_id,
                    #     "content": document,
                    #     "metadata": metadata,
                    #     "distance": distance,
                    #     "rank": i + 1,
                    # })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
                
            return retrieved_docs    
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return[]
        
rag_retriever=RAGRetriever(vectorStore, embedding_manager)
        

In [27]:
rag_retriever

In [ ]:
rag_retriever.retrieve("what is my first skill that i mention in ruseme?")

### Integration Vectordb Context pipeline With LLM output

In [29]:
### 1. Simple RAG pipeline with groq LLM

from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY is missing from your .env file")

llm = ChatGroq(api_key=groq_api_key, model="openai/gpt-oss-20b", temperature=0.1, max_tokens=500)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retriever the context
    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
       return "No relevant context found to answer the question."
    
    ## generate the response using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely and accurately elaborate little bit. Context: {context} Question: {query} Answer: """ 
    
    response = llm.invoke(prompt.format(context=context, query=query))
    return response.content if response else "No response generated by the LLM."

In [30]:
answer=rag_simple("what is my first project that i mention in ruseme?", rag_retriever, llm, top_k=3)
print(answer)

Retrieving documents for query: 'what is my first project that i mention in ruseme?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 54.63it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**Answer:**  
Your first project mentioned in the résumé is **“Ai‑side‑builder”** – a platform that helps users turn ideas into real websites in minutes.


### Enhanced RAG Pipeline Features

In [ ]:
#--- Enhanced RAG Pipeline Features
def rag_advanced(query, retriever, llm, top_k=5 , min_score=0.2, return_context=False):
    """
        RAG pipeline with extra features:
        Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
          'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
          'page': doc['metadata'].get('page', 'unknown'),
          'score': doc['similarity_score'],
          'preview': doc['content'][:300] + '...'
     } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    prompt=f"""Use the following context to answer the question concisely and accurately elaborate little bit. Context: {context} Question: {query} Answer: """ 
    response = llm.invoke(prompt.format(context=context, query=query))
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
        }
    if return_context:
        output['context'] = context
    return output
    
    

In [39]:
result = rag_advanced("what is my first project that i mention in ruseme and explain the project?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is my first project that i mention in ruseme and explain the project?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.35it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **Ai‑side‑builder**

Ai‑side‑builder is the first project you highlighted. It’s an AI‑powered platform that lets users transform a concept into a fully‑functional website in just minutes. You simply describe what you want—layout, style, content—and the system instantly generates a live preview. From there you can tweak the design or content with straightforward requests, and the AI updates the site on the fly. The focus is on speed, creativity, and flexibility, so developers and non‑technical users alike can bring ideas to life without wrestling with code or complex design tools.
Sources: [{'source': '..\\data\\pdf\\Avirup_kar_CV.pdf', 'page': 0, 'score': -0.4236196279525757, 'preview': 'PROJECTS\nAi-side-builder\u2060\n, This platform that helps you turn ideas into real websites in minutes\n•An intelligent platform that helps you turn ideas into real websites in minutes. Just describe what you want, \npreview the result instantly, and refine it through simple requests. Built f

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is my first project that i mention in ruseme and explain the project?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])